# 07 — Async Jobs and Webhooks

`run_area_and_wait` is a wrapper around three lower-level calls:

| Call                                | Purpose                              |
|-------------------------------------|--------------------------------------|
| `client.run_area(payload, polygon)` | Submit jobs, return an `AreaSchedule` |
| `client.check_area_state(schedule)` | Poll job status -> `AreaState`        |
| `client.merge_area_jobs(schedule)`  | Download results -> `AreaResult`      |

If you have something else to do while jobs run (long batch, many
polygons, kicking off in one process and merging in another), use the
three-call form instead of blocking.

This notebook shows both flows, plus the webhook API for
push-notifications.

In [1]:
from dotenv import load_dotenv

load_dotenv()

import time
from infrared_sdk import InfraredClient
from infrared_sdk.analyses.types import AnalysesName, WindModelRequest
from cities import get

city = get("munich")

## 1. The async flow, step by step

In [2]:
with InfraredClient() as client:
    area = client.buildings.get_area(city.polygon_small)

    # 1) Submit. Returns immediately with the schedule.
    schedule = client.run_area(
        WindModelRequest(
            analysis_type=AnalysesName.wind_speed, wind_speed=5, wind_direction=270
        ),
        city.polygon_small,
        buildings=area.buildings,
    )
    print(f"submitted: {len(schedule.jobs)} job(s)")
    print(f"schedule.config_hash : {schedule.config_hash[:16]}...")

    # 2) Poll. AreaState gives you running/succeeded/failed counts.
    while True:
        state = client.check_area_state(schedule)
        print(
            f"  [{state.status}] running={state.running} done={state.succeeded} "
            f"failed={state.failed} of {state.total}"
        )
        if state.is_complete:
            break
        time.sleep(2)

    # 3) Merge. Returns the same AreaResult you'd get from run_area_and_wait.
    result = client.merge_area_jobs(schedule)

print(
    f"\nfinal: jobs {result.succeeded_jobs}/{result.total_jobs}, "
    f"grid={result.grid_shape}"
)

submitted: 1 job(s)
schedule.config_hash : 64ce1aa7b8f27035...


  [pending] running=0 done=0 failed=0 of 1


  [completed] running=0 done=1 failed=0 of 1



final: jobs 1/1, grid=(256, 256)


## 2. Persist a schedule and reload it

`AreaSchedule` is a dataclass with `to_dict()` / `from_dict()`, so you
can submit jobs in one process and merge them in another.

In [3]:
import json

# Save
schedule_json = schedule.to_dict()
print("schedule keys:", list(schedule_json))

# Round-trip
from infrared_sdk.tiling.types import AreaSchedule

restored = AreaSchedule.from_dict(json.loads(json.dumps(schedule_json)))
print(f"restored polygon match : {restored.polygon == schedule.polygon}")
print(f"restored config_hash   : {restored.config_hash[:16]}...")

schedule keys: ['jobs', 'polygon', 'analysis_type', 'config_hash', 'tile_positions', 'grid_shape', 'config']
restored polygon match : True
restored config_hash   : 64ce1aa7b8f27035...


## 3. Webhooks

Instead of polling, register a webhook URL and the dispatcher will POST
status updates as jobs progress. The SDK's webhooks sub-client wraps
the registration / list / delete API.

> Webhooks need a publicly reachable URL. To play locally, run
> `webhook_receiver.py` in this folder (a tiny Flask server) and expose
> it via cloudflared / ngrok. See the docstring at the top of
> `webhook_receiver.py` for one-line instructions.

In [4]:
# Don't actually register here -- registering against a non-public URL
# will be accepted but never deliver. Show the API surface instead.

with InfraredClient() as client:
    existing = client.webhooks.list()
    print(f"existing webhooks: {len(existing)}")
    for w in existing[:3]:
        print(f"  {w}")

existing webhooks: 0


### Registering and using a webhook

```python
# 1. Run webhook_receiver.py and a tunnel, then grab the public URL.
public_url = "https://your-tunnel.example/infrared"

with InfraredClient() as client:
    hook = client.webhooks.register(url=public_url, type="job.completed")
    print(f"registered: {hook}")

    # Submit and get pinged when each tile finishes.
    schedule = client.run_area(
        WindModelRequest(analysis_type=AnalysesName.wind_speed,
                         wind_speed=5, wind_direction=270),
        city.polygon_small,
        buildings=area.buildings,
        webhook_url=public_url,
        webhook_events=["job.completed", "job.failed"],
    )
    # ... receiver prints each status update ...
    result = client.merge_area_jobs(schedule)

    # Cleanup
    client.webhooks.delete(hook["id"])
```

### Verifying webhook signatures

The dispatcher signs each webhook so you can confirm it really came
from Infrared. The receiver should call `client.webhooks.verify_signature(...)`
(see SDK README for the exact signature scheme) before trusting the body.

That's the full async surface. Pair with `04_tiling_and_area_api.ipynb`
for the polygon side and `05_analysis_types_tour.ipynb` for the
payload catalogue.